In [188]:
from random import *
import random

In [235]:
class Maze:
    """
    Classe Labyrinthe
    Représentation sous forme de graphe non-orienté
    dont chaque sommet est une cellule (un tuple (l, c))
    et dont la structure est représentée par un dictionnaire
      - clés : sommets
      - valeurs : ensemble des sommets voisins accessibles
    """
    def __init__(self, height, width):
        """
        Constructeur d'un labyrinthe de height cellules de haut 
        et de width cellules de large 
        Les voisinages sont initialisés à des ensembles vides
        Remarque : dans le labyrinthe créé, chaque cellule est complètement emmurée
        """
        self.height    = height
        self.width     = width
        self.neighbors = {(i,j): set() for i in range(height) for j in range (width)}

    def info(self):
        """
        **NE PAS MODIFIER CETTE MÉTHODE**
        Affichage des attributs d'un objet 'Maze' (fonction utile pour deboguer)
        Retour :
            chaîne (string) : description textuelle des attributs de l'objet
        """
        global c1, c2
        txt = "**Informations sur le labyrinthe**\n"
        txt += f"- Dimensions de la grille : {self.height} x {self.width}\n"
        txt += "- Voisinages :\n"
        txt += str(self.neighbors)+"\n"
        valid = True
        for c1 in {(i, j) for i in range(self.height) for j in range(self.width)}:
            for c2 in self.neighbors[c1]:
                if c1 not in self.neighbors[c2]:
                    valid = False
                    break
            else:
                continue
            break
        txt += "- Structure cohérente\n" if valid else f"- Structure incohérente : {c1} X {c2}\n"
        return txt

    def __str__(self):
        """
        **NE PAS MODIFIER CETTE MÉTHODE**
        Représentation textuelle d'un objet Maze (en utilisant des caractères ascii)
        Retour :
             chaîne (str) : chaîne de caractères représentant le labyrinthe
        """
        txt = ""
        # Première ligne
        txt += "┏"
        for j in range(self.width-1):
            txt += "━━━┳"
        txt += "━━━┓\n"
        txt += "┃"
        for j in range(self.width-1):
            txt += "   ┃" if (0,j+1) not in self.neighbors[(0,j)] else "    "
        txt += "   ┃\n"
        # Lignes normales
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += "   ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else "    "
            txt += "\n"
        # Bas du tableau
        txt += "┗"
        for i in range(self.width-1):
            txt += "━━━┻"
        txt += "━━━┛\n"

        return txt

    def add_wall(self, c1, c2):
        # Facultatif : on teste si les sommets sont bien dans le labyrinthe
        assert 0 <= c1[0] < self.height and \
            0 <= c1[1] < self.width and \
            0 <= c2[0] < self.height and \
            0 <= c2[1] < self.width, \
            f"Erreur lors de l'ajout d'un mur entre {c1} et {c2} : les coordonnées de sont pas compatibles avec les dimensions du labyrinthe"
        # Ajout du mur
        if c2 in self.neighbors[c1]:      # Si c2 est dans les voisines de c1
            self.neighbors[c1].remove(c2) # on le retire
        if c1 in self.neighbors[c2]:      # Si c3 est dans les voisines de c2
            self.neighbors[c2].remove(c1) # on le retire

    def remove_wall(self,c1, c2):
        # Facultatif : on teste si les sommets sont bien dans le labyrinthe
        assert 0 <= c1[0] < self.height and \
            0 <= c1[1] < self.width and \
            0 <= c2[0] < self.height and \
            0 <= c2[1] < self.width, \
            f"Erreur lors du retrait d'un mur entre {c1} et {c2} : les coordonnées de sont pas compatibles avec les dimensions du labyrinthe"
        # Retrait du mur
        if c2 not in self.neighbors[c1]:
            self.neighbors[c1].add(c2) # on l'ajoute
        if c1 not in self.neighbors[c2]:
            self.neighbors[c2].add(c1) # on l'ajoute
            
    def get_all_cells(self):
        liste=[]
        for i in range (0,self.height):
            for j in range (0,self.width):
                liste.append((i, j))
        return liste

    def empty(self):
        for i in range (self.height):
            for j in range (self.width):
                if j+1<self.width:
                    self.remove_wall((i,j),(i,j+1))
                if i+1<self.height:
                    self.remove_wall((i,j),(i+1,j))

    def get_all_walls(self)->list:
        liste=[]
        for i in range (self.height):
            for j in range (self.width):

                if i+1<self.height and (i+1,j) not in (self.neighbors[(i,j)]):
                    liste.append([(i, j), (i + 1, j)])
                if j+1<self.width and (i,j+1) not in (self.neighbors[(i,j)]):
                    liste.append([(i, j), (i, j + 1)])
        return liste

    def get_contiguous_cells(self,c)->list:
        coX,coY=c[0],c[1]
        liste=[]
        trouve=True
        if coX>self.height-1 or coY>self.width-1:
            trouve=False
        if trouve:
            if coX-1>=0:
                liste.append((coX - 1, coY))
            if coX+1<self.height:
                liste.append((coX + 1, coY))
            if coY-1>=0:
                liste.append((coX, coY - 1))
            if coY+1<self.width:
                liste.append((coX, coY + 1))
        else:
            print("La cellune n'existe pas")
        return liste

    def get_reachable_cells(self, c)->list:
        coX,coY=c[0],c[1]

        trouve=True
        if coX>self.height-1 or coY>self.width-1:
            trouve=False
        liste=[]
        if trouve:
            liste_contiguous=self.get_contiguous_cells(c)
            for z in liste_contiguous:
                if z in self.neighbors[c] and c in self.neighbors[z]:
                    liste.append(z)
        else:
            print("La cellune n'existe pas")
        return liste

    @classmethod
    def gen_btree(cls,h : int, w : int):
        lab=Maze(h,w)
        for i in lab.neighbors:
            coX,coY=i[0],i[1]
            voisins=lab.get_reachable_cells(i)
            if coY+1<lab.width and coX+1<lab.height:
                if (coX,coY+1) not in voisins and (coX+1,coY) not in voisins:
                    aleatoire=randint(0,1)
                    if aleatoire==1:
                        lab.remove_wall(i,(coX,coY+1))
                    else:
                        lab.remove_wall(i,(coX+1,coY))
            elif coX+1<lab.height :
                if (coX+1,coY) not in voisins:
                    lab.remove_wall(i,(coX+1,coY))
            elif coY+1<lab.width:
                if (coX,coY+1) not in voisins:
                    lab.remove_wall(i,(coX,coY+1))
        return lab

    @classmethod
    def gen_sidewinder(cls,h : int, w : int):
        lab=Maze(h,w)
        for i in range(lab.height-1):
            sequence=[]
            j=0
            for j in range(lab.width-1):
                sequence.append((i,j))
                aleatoire=randint(0,1)
                if aleatoire==1:
                    lab.remove_wall((i,j),(i,j+1))
                elif aleatoire==0:
                    aleatoire2=randint(0,len(sequence)-1)
                    coX,coY=sequence[aleatoire2]
                    lab.remove_wall((coX,coY),(coX+1,coY))
                    sequence=[]
            sequence.append((i,j+1))
            coX,coY=sequence[randint(0,len(sequence)-1)]
            lab.remove_wall((coX,coY),(coX+1,coY))
        i=lab.height-1
        for j in range(lab.width-1):
            lab.remove_wall((i,j),(i,j+1))
        return lab

    @classmethod
    def gen_fusion(cls, h : int, w : int):
        lab=Maze(h,w)
        #Labélisation des cellules de 1 à n.
        dico={}
        j=1
        for i in lab.neighbors:
            dico[i]=[j]
            j+=1

        #On extrait la liste de tous les murs et on les « mélange ».
        liste_mur=lab.get_all_walls()
        shuffle(liste_mur)

        for i in range(len(liste_mur)):
            c1=liste_mur[i][0]
            c2=liste_mur[i][1]
            indice1=dico[c1]
            indice2=dico[c2]

            if indice1 != indice2:
                lab.remove_wall(c1,c2)
                for j in dico:
                    if dico[j]==indice2:
                        dico[j]=indice1
        return lab

    @classmethod
    def gen_exploration(cls,h: int, w: int):
        lab=Maze(h,w)
        #choix d'une cellule au hasard :
        cellule_depart = (random.randint(0, h-1), random.randint(0, w-1))
        #on la renseigne comme "visitée".
        visitee = {cellule_depart}
        #on ajoute la cellule dans une pile
        pile = [cellule_depart]

        #tant que la pile n'est pas vide
        while pile:

            #on prend la cellule en haut de la pile et on l'enlève
            cel_actuelle = pile[-1]
            voisins = lab.get_contiguous_cells(cel_actuelle)

            # on filtre les voisins non visités
            voisins_non_visites = []
            for voisin in voisins:
                if voisin not in visitee:
                    voisins_non_visites.append(voisin)

            if voisins_non_visites:
                # on choisit un voisin au hasard
                cel_suivante = random.choice(voisins_non_visites)

                # on casse le mur entre la cellule courante et la cellule voisine
                lab.remove_wall(cel_actuelle, cel_suivante)

                # on marque la cellule voisine comme visitée et l'ajouter à la pile
                visitee.add(cel_suivante)
                pile.append(cel_suivante)
            else:
                # si la cellule n'a pas de voisins non visités, on la retire de la pile
                pile.pop()

        return lab

    @classmethod
    def gen_wilson(cls,h,w):
        #je crée le labyrinthe avec tous les murs
        lab=Maze(h,w)
        #enregistre une liste de toutes les cellules du labyrinthe dans une variable
        toutes_cellulles=lab.get_all_cells()
        #selectionne la cellule de départ de la génération depuis la liste des cellules et la marque
        cellule_depart=toutes_cellulles.pop(randint(0,len(toutes_cellulles)-1))
        marque=[cellule_depart]

        #Tant que ma liste de cellules n'est pas vide (tant qu'on à pas utilisé toutes les cellules)
        while len(toutes_cellulles)!=0:
            #je sélectionne la première cellule du chemin et la retire de la liste des cellules
            cellule = toutes_cellulles.pop(randint(0,len(toutes_cellulles)-1))
            #initialise un booleen qui nous permet de vérifier si le chemin est fini
            bon_chemin=False
            passage=[]
            #tant que le chemin n'est pas fini
            while not bon_chemin:
                passage= [cellule]
                c=cellule
                #tant que la dernière cellule du chemin n'est pas marqué…
                while passage[-1] not in marque:
                    #je sélectionne une cellule parmi les cellules voisines à la précédente utilisée
                    accessible=lab.get_contiguous_cells(c)
                    c=accessible[randint(0,len(accessible)-1)]
                    #si elle crée une boucle, je la supprime ainsi que tous les éléments présents depuis la première présence de la cellule
                    if c in passage:
                        passage = passage[:passage.index(c)+1]
                    #sinon je l'ajoute à mon chemin
                    else:
                        passage.append(c)
                    #si c est marqué
                    if c in marque:
                        #mon chemin est fini.
                        bon_chemin=True
                #j'ajoute cellule au marquage
                marque.append(cellule)
            #j'initialise une liste qui va me permettre de retrouver les indices de chaque cellule du chemin dans toutes_cellulles et je supprime les murs entre les différents éléments
            liste=[]
            for i in range (0,len(passage)-1):
                lab.remove_wall(passage[i],passage[i+1])
                for j in range(0,len(toutes_cellulles)):
                    if passage[i]==toutes_cellulles[j]:
                        liste.append(j)
            #je trie ma cellule dans l'ordre décroissant en partant du plus grand indice car comme je .pop les éléments ensuite, si elle était dans l'ordre croissant las indices changerais
            liste.sort(reverse=True)
            for i in liste:
                #je marque les cellules qui constituent le chemin
                marque.append(toutes_cellulles.pop(i))
        marque.sort()
        return lab

    def overlay(self, content=None):
        """
        Rendu en mode texte, sur la sortie standard, \
        d'un labyrinthe avec du contenu dans les cellules
        Argument :
        content (dict) : dictionnaire tq content[cell] contient le caractère à afficher au milieu de la cellule
        Retour :
        string
        """
        if content is None:
            content = {(i,j):' ' for i in range(self.height) for j in range(self.width)}
        else:
            # Python >=3.9
            #content = content | {(i, j): ' ' for i in range(
            #    self.height) for j in range(self.width) if (i,j) not in content}
            # Python <3.9
            new_content = {(i, j): ' ' for i in range(self.height) for j in range(self.width) if (i,j) not in content}
            content = {**content, **new_content}
        txt = r""
        # Première ligne
        txt += "┏"
        for j in range(self.width-1):
            txt += "━━━┳"
        txt += "━━━┓\n"
        txt += "┃"
        for j in range(self.width-1):
            txt += " "+content[(0,j)]+" ┃" if (0,j+1) not in self.neighbors[(0,j)] else " "+content[(0,j)]+"  "
        txt += " "+content[(0,self.width-1)]+" ┃\n"
        # Lignes normales
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += " "+content[(i+1,j)]+" ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else " "+content[(i+1,j)]+"  "
            txt += "\n"
        # Bas du tableau
        txt += "┗"
        for i in range(self.width-1):
            txt += "━━━┻"
        txt += "━━━┛\n"
        return txt

    def solve_dfs(self,start, stop):
        #Parcours du graphe jusqu’à ce qu’on trouve A
        marque=[start]
        pile=[start]
        pred= {start: {start}}
        trouveA=False
        while len(pile) !=0 and trouveA==False:
            c=pile.pop(0)
            if c==stop:
                trouveA=True
            else:
                for c2 in self.get_reachable_cells(c):
                    if c2 not in marque:
                        marque.append(c2)
                        pile=[c2]+pile
                        pred[c2]=c
        #Reconstruction du chemin à partir des prédécesseurs
        c=stop
        chemin=[]
        while c!=start:
            chemin.append(c)
            a=pred[c]
            c=a
        chemin.append(start)
        return chemin

    def solve_bfs(self,start, stop):
        #Parcours du graphe jusqu’à ce qu’on trouve A
        marque=[start]
        file=[start]
        pred= {start: {start}}
        trouveA=False
        while len(file) !=0 and trouveA==False:
            c=file.pop(0)
            if c==stop:
                trouveA=True
            else:
                for c2 in self.get_reachable_cells(c):
                    if c2 not in marque:
                        marque.append(c2)
                        file.append(c2)
                        pred[c2]=c
        #Reconstruction du chemin à partir des prédécesseurs
        c=stop
        chemin=[]
        while c!=start:
            chemin.append(c)
            a=pred[c]
            c=a
        chemin.append(start)
        return chemin

    def solve_rhr(self,start, stop):
        chemin=[start]
        c=start
        coX,coY=c[0],c[1]
        reachable=self.get_reachable_cells(c)
        #Détermine la première case vers lequel je me dirige
        if (coX+1,coY) in reachable and (coX,coY-1) not in reachable:
            c=(coX+1,coY)
        elif (coX,coY+1) in reachable and (coX+1,coY) not in reachable:
            c=(coX,coY+1)
        elif (coX-1,coY) in reachable and (coX,coY+1) not in reachable:
            c=(coX-1,coY)
        elif (coX-1,coY) in reachable and (coX-1,coY) not in reachable:
            c=(coX,coY+1)
        else:
            c=(coX+1,coY)
        trouveA=False
        while not trouveA:
            if c==stop:
                trouveA=True
                chemin.append(c)
            else:
                origine=chemin[len(chemin)-1]
                coX,coY=c[0],c[1]
                reachable=self.get_reachable_cells(c)
                #si je viens de la case du dessus
                if (origine[0],origine[1]) ==(coX-1,coY):
                    #ma case de gauche est-elle accessible ?
                    if (coX,coY-1) in reachable:
                        chemin.append(c)
                        c=(coX,coY-1)
                    #ma case de dessous est-elle accessible ?
                    elif (coX+1,coY) in reachable:
                        chemin.append(c)
                        c=(coX+1,coY)
                    #ma case de droite est-elle accessible ?
                    elif (coX,coY+1) in reachable:
                        chemin.append(c)
                        c=(coX,coY+1)
                        #je remonte
                    else:
                        chemin.append(c)
                        c=(coX-1,coY)
                #si je viens de la case de gauche
                elif (origine[0],origine[1]) ==(coX,coY-1):
                    #ma case de dessous est-elle accessible ?
                    if (coX+1,coY) in reachable:
                        chemin.append(c)
                        c=(coX+1,coY)
                    #ma case de droite est-elle accessible ?
                    elif (coX,coY+1) in reachable:
                        chemin.append(c)
                        c=(coX,coY+1)
                    #ma case du dessus est-elle accessible ?
                    elif (coX-1,coY) in reachable:
                        chemin.append(c)
                        c=(coX-1,coY)
                    #je retourne à gauche
                    else:
                        chemin.append(c)
                        c=(coX,coY-1)
                #si je viens de la case du bas
                elif (origine[0],origine[1]) ==(coX+1,coY):
                    #ma case de droite est-elle accessible ?
                    if (coX,coY+1) in reachable:
                        chemin.append(c)
                        c=(coX,coY+1)
                    #ma case du dessus est-elle accessible ?
                    elif (coX-1,coY) in reachable:
                        chemin.append(c)
                        c=(coX-1,coY)
                    #ma case de gauche est-elle accessible ?
                    elif (coX,coY-1) in reachable:
                        chemin.append(c)
                        c=(coX,coY-1)
                    #je redescend
                    else:
                        chemin.append(c)
                        c=(coX+1,coY)
                #si je viens de la case de droite
                else:
                    #ma case du dessus est-elle accessible ?
                    if (coX-1,coY) in reachable:
                        chemin.append(c)
                        c=(coX-1,coY)
                    #ma case de gauche est-elle accessible ?
                    elif (coX,coY-1) in reachable:
                        chemin.append(c)
                        c=(coX,coY-1)
                    #ma case de dessous est-elle accessible ?
                    elif (coX+1,coY) in reachable:
                        chemin.append(c)
                        c=(coX+1,coY)
                    #je retourne à droite
                    else:
                        chemin.append(c)
                        c=(coX,coY+1)
        return chemin

    def distance_geo(self,c1, c2):
        chemin=self.solve_bfs(c1, c2)
        #on crée une variable chemin qui est une liste des coordonnées des différentes étapes qu'on a traversées entre c1 et c2.
        deplacement=len(chemin)-1
        #on calcule le nombre de déplacements, cette variabl est calculée en prenant la longueur du chemin moins 1.
        return deplacement

    def distance_man(self,c1, c2):
        # Facultatif : on teste si les sommets sont bien dans le labyrinthe
        assert 0 <= c1[0] < self.height and \
            0 <= c1[1] < self.width and \
            0 <= c2[0] < self.height and \
            0 <= c2[1] < self.width, \
            f"Erreur lors de la vérification de la distance entre {c1} et {c2} : les coordonnées de sont pas compatibles avec les dimensions du labyrinthe"
        deplacement=0
         # On initialise la variable "deplacement" à 0, cette variable comptera le nombre de déplacements nécessaires
        coX,coY=c1[0],c1[1]
        # On décompose les coordonnées de c1 dans les variables coX et coY 
        while coX != c2[0] and coX<self.height:
        # Tant que la coordonnée X de c1 n'est pas égale à la coordonnée X de c2 (c'est-à-dire tant que l'on n'est pas arrivé à la même ligne).
            while coY != c2[1] and coY<self.height:
            # Tant que la coordonnée Y de c1 n'est pas égale à celle de c2 (c'est-à-dire tant que l'on n'est pas arrivé à la même colonne).
                coY=coY+1
                # On incrémente la coordonnée Y de 1 pour avancer d'une colonne
                deplacement+=1
                # À chaque déplacement, on incrémente le nombre de déplacements effectués
            coX=coX+1
             # Une fois que la colonne est atteinte, on incrémente la coordonnée X pour avancer d'une ligne
            deplacement+=1
            # On incrémente aussi le nombre de déplacements à chaque déplacement sur la ligne
        return deplacement
    
    
    
    def worst_path_len(self)->int:
        """
        Retourne la longueur du plus long chemin partant du départ à une impasse (feuille).
        """
        #j'initialise la cellule de départ à (0,0) et la longueur max à 0
        cellule_depart=(0,0)
        max_longueur=0
        #je parcours toutes les cellules
        for cellule in self.neighbors:
            #si la longueur de la distance géodésique entre ma cellule de départ et ma cellule actuelle est plus grande que la précédente
            if self.distance_geo(cellule_depart,cellule) > max_longueur:
                #je modifie la longueur max en la longueur de la distance géodésique entre ma cellule de départ et ma cellule actuelle
                max_longueur=self.distance_geo(cellule_depart,cellule)
        #je retourne ma longueur
        return max_longueur
    
    
    def dead_end_number(self):
        """
        Calcul du nombre de culs-de-sac dans le labyrinthe.
        """
        dead_ends = 0
        #on renseigne une variable dead_ends qui regroupera tous les culs de sac

        for i in range(self.height):
            for j in range(self.width):
                #parcours de chaque lignes et colonnes
                cellule = (i, j)
                #définition d'une variable cellule associée a un tuple qui regroupe les coordonnées de la cellule actuelle
                if cellule in self.neighbors:
                    #si notre cellule est concernée par l'attribut neighbours (qu'elle a des voisins) on rentre dans le if: 
                    if len(self.neighbors[cellule]) == 1:
                        #on vérifie si la cellule n'a qu'un seul voisin accessible (qu'elle n'a qu'une seule entrée)
                        dead_ends += 1
                        #si la case est concernée on ajoute 1 a notre variable
        
        return dead_ends


In [236]:
laby = Maze(4, 4)
print(laby.info())

**Informations sur le labyrinthe**
- Dimensions de la grille : 4 x 4
- Voisinages :
{(0, 0): set(), (0, 1): set(), (0, 2): set(), (0, 3): set(), (1, 0): set(), (1, 1): set(), (1, 2): set(), (1, 3): set(), (2, 0): set(), (2, 1): set(), (2, 2): set(), (2, 3): set(), (3, 0): set(), (3, 1): set(), (3, 2): set(), (3, 3): set()}
- Structure cohérente



In [191]:
print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┛



In [192]:
laby.neighbors = {
    (0, 0): {(1, 0)},
    (0, 1): {(0, 2), (1, 1)},
    (0, 2): {(0, 1), (0, 3)},
    (0, 3): {(0, 2), (1, 3)},
    (1, 0): {(2, 0), (0, 0)},
    (1, 1): {(0, 1), (1, 2)},
    (1, 2): {(1, 1), (2, 2)},
    (1, 3): {(2, 3), (0, 3)},
    (2, 0): {(1, 0), (2, 1), (3, 0)},
    (2, 1): {(2, 0), (2, 2)},
    (2, 2): {(1, 2), (2, 1)},
    (2, 3): {(3, 3), (1, 3)},
    (3, 0): {(3, 1), (2, 0)},
    (3, 1): {(3, 2), (3, 0)},
    (3, 2): {(3, 1)},
    (3, 3): {(2, 3)}
}

print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃
┣   ╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋   ┫
┃           ┃   ┃
┣   ╋━━━╋━━━╋   ┫
┃           ┃   ┃
┗━━━┻━━━┻━━━┻━━━┛



In [193]:
laby.neighbors[(1,3)].remove((2,3))
laby.neighbors[(2,3)].remove((1,3))
print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃
┣   ╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋━━━┫
┃           ┃   ┃
┣   ╋━━━╋━━━╋   ┫
┃           ┃   ┃
┗━━━┻━━━┻━━━┻━━━┛



In [194]:
laby.neighbors[(1, 3)].add((2, 3))
laby.neighbors[(2, 3)].add((1, 3))
print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃
┣   ╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋   ┫
┃           ┃   ┃
┣   ╋━━━╋━━━╋   ┫
┃           ┃   ┃
┗━━━┻━━━┻━━━┻━━━┛



In [195]:
laby.neighbors[(1, 3)].remove((2, 3))
print(laby)
print(laby.info())

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃
┣   ╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋━━━┫
┃           ┃   ┃
┣   ╋━━━╋━━━╋   ┫
┃           ┃   ┃
┗━━━┻━━━┻━━━┻━━━┛

**Informations sur le labyrinthe**
- Dimensions de la grille : 4 x 4
- Voisinages :
{(0, 0): {(1, 0)}, (0, 1): {(1, 1), (0, 2)}, (0, 2): {(0, 1), (0, 3)}, (0, 3): {(0, 2), (1, 3)}, (1, 0): {(2, 0), (0, 0)}, (1, 1): {(0, 1), (1, 2)}, (1, 2): {(1, 1), (2, 2)}, (1, 3): {(0, 3)}, (2, 0): {(1, 0), (2, 1), (3, 0)}, (2, 1): {(2, 0), (2, 2)}, (2, 2): {(1, 2), (2, 1)}, (2, 3): {(3, 3), (1, 3)}, (3, 0): {(3, 1), (2, 0)}, (3, 1): {(3, 2), (3, 0)}, (3, 2): {(3, 1)}, (3, 3): {(2, 3)}}
- Structure incohérente : (2, 3) X (1, 3)



In [196]:
laby.neighbors[(2, 3)].remove((1,3))

In [197]:
c1 = (1, 3)
c2 = (2, 3)
if c1 in laby.neighbors[c2] and c2 in laby.neighbors[c1]:
    print(f"Il n'y a pas de mur entre {c1} et {c2} car elles sont mutuellement voisines")
elif c1 not in laby.neighbors[c2] and c2 not in laby.neighbors[c1]:
    print(f"Il y a un mur entre {c1} et {c2} car {c1} n'est pas dans le voisinage de {c2} et {c2} n'est pas dans le voisinage de {c1}")
else:
    print(f"Il y a une incohérence de réciprocité des voisinages de {c1} et {c2}")

Il y a un mur entre (1, 3) et (2, 3) car (1, 3) n'est pas dans le voisinage de (2, 3) et (2, 3) n'est pas dans le voisinage de (1, 3)


In [198]:
c1 = (1, 3)
c2 = (2, 3)
if c1 in laby.neighbors[c2] and c2 in laby.neighbors[c1]:
    print(f"{c1} est accessible depuis {c2} et vice-versa")
elif c1 not in laby.neighbors[c2] and c2 not in laby.neighbors[c1]:
    print(f"{c1} n'est pas accessible depuis {c2} et vice-versa")
else:
    print(f"Il y a une incohérence de réciprocité des voisinages de {c1} et {c2}")

(1, 3) n'est pas accessible depuis (2, 3) et vice-versa


In [199]:
L = []
for i in range(laby.height):
    for j in range(laby.width):
        L.append((i,j))
print(f"Liste des cellules : \n{L}")

Liste des cellules : 
[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3), (3, 0), (3, 1), (3, 2), (3, 3)]


In [200]:
laby = Maze(1, 2)
print(laby)

┏━━━┳━━━┓
┃   ┃   ┃
┗━━━┻━━━┛



In [201]:
laby.neighbors[(0, 0)].add((0, 1))
laby.neighbors[(0, 1)].add((0, 0))
print(laby)

┏━━━┳━━━┓
┃       ┃
┗━━━┻━━━┛



In [202]:
laby.add_wall((0,0), (0,1))
print(laby)

┏━━━┳━━━┓
┃   ┃   ┃
┗━━━┻━━━┛



In [203]:
laby = Maze(5, 5)
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛



In [204]:
laby.remove_wall((0, 0), (0, 1))
print (laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃       ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛



In [205]:
print(laby.get_all_cells())

[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]


In [206]:
laby.empty()
laby.add_wall((0, 0), (0, 1))
laby.add_wall((0, 1), (1, 1))
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃               ┃
┣   ╋━━━╋   ╋   ╋   ┫
┃                   ┃
┣   ╋   ╋   ╋   ╋   ┫
┃                   ┃
┣   ╋   ╋   ╋   ╋   ┫
┃                   ┃
┣   ╋   ╋   ╋   ╋   ┫
┃                   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛



In [207]:
print(laby.get_all_walls())

[[(0, 0), (0, 1)], [(0, 1), (1, 1)]]


In [208]:
print(laby.get_contiguous_cells((4,4)))

[(3, 4), (4, 3)]


In [209]:
print(laby.get_reachable_cells((0,1)))

[(0, 2)]


In [210]:
laby=Maze.gen_btree(4, 4)
print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋   ┫
┃           ┃   ┃
┣━━━╋━━━╋   ╋   ┫
┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋   ┫
┃               ┃
┗━━━┻━━━┻━━━┻━━━┛



In [211]:
laby = Maze.gen_sidewinder(4, 4)
print(laby)

┏━━━┳━━━┳━━━┳━━━┓
┃       ┃   ┃   ┃
┣   ╋━━━╋   ╋   ┫
┃               ┃
┣   ╋━━━╋━━━╋━━━┫
┃   ┃           ┃
┣   ╋━━━╋   ╋━━━┫
┃               ┃
┗━━━┻━━━┻━━━┻━━━┛



In [212]:
laby = Maze.gen_fusion(15,15)
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃                   ┃       ┃           ┃   ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ┫
┃   ┃   ┃       ┃   ┃       ┃       ┃   ┃           ┃       ┃
┣   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━┫
┃                   ┃       ┃       ┃           ┃       ┃   ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃   ┃               ┃   ┃           ┃           ┃           ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━┫
┃   ┃                               ┃   ┃               ┃   ┃
┣━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃               ┃                           ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ┫
┃       ┃           ┃   ┃       ┃   ┃   ┃   ┃               ┃
┣━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃       ┃   ┃               ┃       ┃   ┃           ┃   ┃   ┃
┣━━━╋   

In [213]:
laby = Maze.gen_exploration(15,15)
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃               ┃                   ┃                       ┃
┣   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋━━━╋━━━╋   ┫
┃   ┃           ┃   ┃           ┃   ┃   ┃       ┃           ┃
┣━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃       ┃   ┃       ┃   ┃               ┃   ┃       ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━┫
┃   ┃       ┃   ┃   ┃   ┃   ┃       ┃               ┃       ┃
┣   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ┫
┃                       ┃   ┃       ┃           ┃           ┃
┣━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ┫
┃                       ┃       ┃   ┃   ┃   ┃       ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃   ┃               ┃       ┃       ┃   ┃           ┃       ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━┫
┃       ┃           ┃               ┃       ┃   ┃   ┃       ┃
┣   ╋   

In [214]:
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃               ┃                   ┃                       ┃
┣   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋━━━╋━━━╋   ┫
┃   ┃           ┃   ┃           ┃   ┃   ┃       ┃           ┃
┣━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃       ┃   ┃       ┃   ┃               ┃   ┃       ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━┫
┃   ┃       ┃   ┃   ┃   ┃   ┃       ┃               ┃       ┃
┣   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ┫
┃                       ┃   ┃       ┃           ┃           ┃
┣━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ┫
┃                       ┃       ┃   ┃   ┃   ┃       ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃   ┃               ┃       ┃       ┃   ┃           ┃       ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━┫
┃       ┃           ┃               ┃       ┃   ┃   ┃       ┃
┣   ╋   

In [215]:
laby = Maze.gen_wilson(12, 12)
print(laby)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃                   ┃   ┃           ┃           ┃
┣━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋   ╋   ┫
┃   ┃   ┃       ┃   ┃       ┃   ┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━┫
┃           ┃       ┃               ┃   ┃   ┃   ┃
┣   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ┫
┃   ┃                       ┃   ┃       ┃       ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃       ┃       ┃   ┃   ┃       ┃               ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃   ┃   ┃   ┃   ┃       ┃   ┃       ┃   ┃   ┃   ┃
┣   ╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ┫
┃       ┃       ┃   ┃   ┃       ┃       ┃       ┃
┣━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━┫
┃   ┃   ┃   ┃       ┃   ┃   ┃       ┃           ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━┫
┃                       ┃       ┃       ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃       ┃   ┃   ┃   ┃   ┃   ┃       ┃   ┃       ┃


In [216]:
laby = Maze(4,4)
laby.empty()
print(laby.overlay({
    (0, 0):'c',
    (0, 1):'o',
    (1, 1):'u',
    (2, 1):'c',
    (2, 2):'o',
    (3, 2):'u',
    (3, 3):'!'}))

┏━━━┳━━━┳━━━┳━━━┓
┃ c   o         ┃
┣   ╋   ╋   ╋   ┫
┃     u         ┃
┣   ╋   ╋   ╋   ┫
┃     c   o     ┃
┣   ╋   ╋   ╋   ┫
┃         u   ! ┃
┗━━━┻━━━┻━━━┻━━━┛



In [217]:
laby = Maze(4,4)
laby.empty()
path = {(0, 0): '@',
        (1, 0): '*',
        (1, 1): '*',
        (2, 1): '*',
        (2, 2): '*',
        (3, 2): '*',
        (3, 3): '§'}
print(laby.overlay(path))

┏━━━┳━━━┳━━━┳━━━┓
┃ @             ┃
┣   ╋   ╋   ╋   ┫
┃ *   *         ┃
┣   ╋   ╋   ╋   ┫
┃     *   *     ┃
┣   ╋   ╋   ╋   ┫
┃         *   § ┃
┗━━━┻━━━┻━━━┻━━━┛



In [218]:
laby = Maze.gen_fusion(15, 15)
solution = laby.solve_dfs((0, 0), (14, 14))
str_solution = {c:'*' for c in solution}
str_solution[( 0,  0)] = 'D'
str_solution[(14, 14)] = 'A'
print(laby.overlay(str_solution))

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D ┃   ┃       ┃               ┃   ┃       ┃   ┃           ┃
┣   ╋   ╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃ *   * ┃   ┃           ┃   ┃               ┃   ┃   ┃       ┃
┣━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ╋   ┫
┃   ┃ *   * ┃   ┃           ┃       ┃               ┃   ┃   ┃
┣   ╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━┫
┃         *   * ┃           ┃     *   *   *   * ┃   ┃       ┃
┣━━━╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ┫
┃             *   *   *   * ┃     *         ┃ *   *         ┃
┣━━━╋   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ┫
┃       ┃   ┃       ┃   ┃ *   *   * ┃       ┃   ┃ * ┃       ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ┫
┃           ┃           ┃   ┃   ┃   ┃           ┃ *     ┃   ┃
┣   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ┫
┃       ┃   ┃           ┃   ┃       ┃   ┃       ┃ * ┃   ┃   ┃
┣━━━╋━━━

In [219]:
laby = Maze.gen_exploration(15, 15)
solution = laby.solve_bfs((0, 0), (14, 14))
str_solution = {c:'*' for c in solution}
str_solution[( 0,  0)] = 'D'
str_solution[(14, 14)] = 'A'
print(laby.overlay(str_solution))

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D                                     ┃               ┃   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃ * ┃                       ┃               ┃           ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ┫
┃ * ┃ *   * ┃ *   * ┃   ┃   ┃   ┃           ┃       ┃       ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ┫
┃ * ┃ * ┃ * ┃ * ┃ * ┃   ┃                       ┃   ┃       ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ┫
┃ *   * ┃ *   * ┃ * ┃           ┃           ┃       ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋━━━┫
┃ *   *         ┃ *   * ┃   ┃   ┃               ┃           ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ┫
┃ * ┃ *   *   *   *   * ┃   ┃       ┃     *   *   * ┃ *   * ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ┫
┃ * ┃       ┃           ┃     *   *   * ┃ *   * ┃ *   * ┃ * ┃
┣   ╋   

In [220]:
laby = Maze.gen_fusion(15, 15)
solution = laby.solve_rhr((0, 0), (14,14))
str_solution = {c:'*' for c in solution}
str_solution[( 0,  0)] = 'D'
str_solution[(14,14)] = 'A'
print(laby.overlay(str_solution))

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *   *     ┃       ┃   ┃           ┃   ┃   ┃           ┃
┣   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━┫
┃ * ┃ *   *   *   *             ┃                           ┃
┣━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋━━━┫
┃ * ┃ * ┃ * ┃ * ┃ *   * ┃           ┃       ┃       ┃       ┃
┣   ╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃ * ┃ *   *   * ┃ *   * ┃           ┃   ┃       ┃           ┃
┣   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━┫
┃ *   * ┃ *   *   *     ┃   ┃               ┃       ┃       ┃
┣   ╋   ╋━━━╋   ╋━━━╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━┫
┃ * ┃ *   *   * ┃   ┃   ┃                   ┃   ┃   ┃       ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ┫
┃ *   * ┃       ┃       ┃       ┃   ┃       ┃       ┃       ┃
┣━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋━━━┫
┃ * ┃ * ┃   ┃       ┃       ┃       ┃   ┃   ┃       ┃       ┃
┣   ╋   

In [221]:
print(laby.distance_geo((0,0),(0,14)))

60


In [222]:
print(laby.distance_man((0,0),(10,10)))

20


In [250]:
laby = Maze.gen_fusion(10, 10)
print(laby)
print(laby.dead_end_number())

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃       ┃   ┃           ┃
┣   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋━━━┫
┃       ┃   ┃   ┃                       ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ┫
┃           ┃           ┃   ┃       ┃   ┃
┣   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃   ┃   ┃           ┃       ┃       ┃   ┃
┣   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃       ┃       ┃   ┃       ┃
┣━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ┫
┃           ┃                       ┃   ┃
┣━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃               ┃   ┃   ┃       ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━┫
┃       ┃   ┃       ┃   ┃   ┃   ┃       ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃   ┃           ┃       ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃       ┃       ┃           ┃           ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛

27


In [252]:
print(laby)
print(laby.worst_path_len())

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃           ┃       ┃   ┃           ┃
┣   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋━━━┫
┃       ┃   ┃   ┃                       ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ┫
┃           ┃           ┃   ┃       ┃   ┃
┣   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃   ┃   ┃           ┃       ┃       ┃   ┃
┣   ╋━━━╋━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃       ┃       ┃   ┃       ┃
┣━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ┫
┃           ┃                       ┃   ┃
┣━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃               ┃   ┃   ┃       ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━┫
┃       ┃   ┃       ┃   ┃   ┃   ┃       ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃   ┃           ┃       ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃       ┃       ┃           ┃           ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛

33
